# Code-Capacity Edge-Weight Trace

This notebook compares plain BP, scalar edge weighting, and random edge weighting on individual half-stabilizer cases.


In [ ]:
import importlib
from math import comb
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm


In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "relay_bp").exists():
            return candidate
    raise FileNotFoundError("Could not locate the relay repo root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from relay_bp.analysis.code_capacity_common import default_export_code_paths

CODE_PATHS = {
    "hgp_625": Path(r"C:\Users\User\Documents\projects-git\giulio\hgp_code_625_25_6_peg.npz"),
    "hgp_225": Path(r"C:\Users\User\Documents\cluster_decoder\cluster_decoder\codes\hgp_code_225.npz"),
    "surface5": Path(r"C:\Users\User\Documents\Code from BPGD\surface5_HxHzLxLz.npz"),
    "surface13": Path(r"C:\Users\User\Documents\Code from BPGD\surface13_HxHzLxLz.npz"),
    "B1": Path(r"C:\Users\User\Documents\projects-git\bpgd_low_llr_and_hybrid\B1_HxHzLxLz.npz"),
    "gross": Path(r"C:\Users\User\Documents\projects-git\bpgd_low_llr_and_hybrid\gross_HxHzLxLz.npz"),
    "two_gross": Path(r"C:\Users\User\Documents\projects-git\bpgd_low_llr_and_hybrid\two_gross_HxHzLxLz.npz"),
}
for fallback_key, fallback_path in default_export_code_paths(REPO_ROOT).items():
    if fallback_key not in CODE_PATHS or not CODE_PATHS[fallback_key].exists():
        CODE_PATHS[fallback_key] = fallback_path

available_code_keys = [key for key, path in CODE_PATHS.items() if Path(path).exists()]
if not available_code_keys:
    raise FileNotFoundError("No code paths are available. Update CODE_PATHS for this machine.")

selected_code = "gross"  # change here
if selected_code not in CODE_PATHS or not Path(CODE_PATHS[selected_code]).exists():
    selected_code = available_code_keys[0]
npz_path = CODE_PATHS[selected_code]

prior_error_rate = 0.10
max_iter = 40
alpha = 1.0
center_values = np.linspace(0.00, 1.00, 11)
width_values = np.linspace(0.00, 2.00, 11)
random_draws_per_point = 2
n_error_samples = 300
base_seed = 0

show_progress = True
show_inner_case_progress = False
selected_row_weights = None
case_limit = None
sample_limit = None

weak_memory_interval = (-0.10, 0.10)
weak_damping_interval = (0.85, 0.95)

assert np.all(np.diff(center_values) >= 0), "center_values must be sorted."
assert np.all(np.diff(width_values) >= 0), "width_values must be sorted."
assert np.all(width_values >= 0.0), "width_values must be non-negative."

trace_center = 0.0
trace_width = 1.0
trace_seed_offsets = [0, 1, 2]
core_seed = 0


In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "relay_bp").exists():
            return candidate
    raise FileNotFoundError("Could not locate the relay repo root from the current working directory.")


REPO_ROOT = globals().get("REPO_ROOT", find_repo_root(Path.cwd()))
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import relay_bp
import relay_bp.analysis.code_capacity_common as code_capacity_common_module
import relay_bp.analysis.code_capacity_edge_weights as code_capacity_edge_weights_module

importlib.reload(code_capacity_common_module)
importlib.reload(code_capacity_edge_weights_module)

from relay_bp.analysis.code_capacity_common import (
    default_export_code_paths,
    decode_with_edge_message_weights,
    decode_with_memory_strengths,
    enumerate_half_stabilizer_cases,
    evaluate_decode_result,
    load_code_capacity_problem,
    make_min_sum_tracer,
    sample_random_error_cases,
)
from relay_bp.analysis.code_capacity_edge_weights import (
    best_edge_weight_rows,
    build_memory_strengths,
    edge_weight_heatmap_rows,
    evaluate_edge_weight_heatmap,
    interval_from_center_width,
)

CODE_PATHS = globals().get("CODE_PATHS", {})
for fallback_key, fallback_path in default_export_code_paths(REPO_ROOT).items():
    if fallback_key not in CODE_PATHS or not Path(CODE_PATHS[fallback_key]).exists():
        CODE_PATHS[fallback_key] = fallback_path
selected_code = globals().get("selected_code", "gross")
if selected_code not in CODE_PATHS or not Path(CODE_PATHS[selected_code]).exists():
    selected_code = next(key for key, path in CODE_PATHS.items() if Path(path).exists())
npz_path = Path(globals().get("npz_path", CODE_PATHS[selected_code]))

prior_error_rate = float(globals().get("prior_error_rate", 0.10))
max_iter = int(globals().get("max_iter", 40))
alpha = float(globals().get("alpha", 1.0))
center_values = np.asarray(globals().get("center_values", np.linspace(0.00, 1.00, 11)), dtype=np.float64)
width_values = np.asarray(globals().get("width_values", np.linspace(0.00, 2.00, 11)), dtype=np.float64)
random_draws_per_point = int(globals().get("random_draws_per_point", 2))
n_error_samples = int(globals().get("n_error_samples", 300))
base_seed = int(globals().get("base_seed", 0))
show_progress = bool(globals().get("show_progress", False))
show_inner_case_progress = bool(globals().get("show_inner_case_progress", False))
selected_row_weights = globals().get("selected_row_weights", None)
case_limit = globals().get("case_limit", None)
sample_limit = globals().get("sample_limit", None)
weak_memory_interval = tuple(globals().get("weak_memory_interval", (-0.10, 0.10)))
weak_damping_interval = tuple(globals().get("weak_damping_interval", (0.85, 0.95)))


def load_css_code(path: Path) -> dict[str, object]:
    problem = load_code_capacity_problem(path, default_prior_error_rate=prior_error_rate)
    return {"problem": problem, "hx": problem.hx, "hz": problem.hz, "lz": problem.lz, "metadata": problem.metadata}


def dynamic_limits(values: np.ndarray) -> tuple[float, float]:
    finite = np.asarray(values, dtype=np.float64)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return (0.0, 1.0)
    vmin = float(finite.min())
    vmax = float(finite.max())
    if np.isclose(vmin, vmax):
        eps = 1e-6 if np.isclose(vmin, 0.0) else max(1e-6, abs(vmin) * 0.05)
        return (vmin - eps, vmax + eps)
    return (vmin, vmax)


def plot_heatmaps(artifact: dict[str, object], *, title: str):
    metrics = [
        ("logical_success_rate", "Logical success rate"),
        ("convergence_rate", "Convergence rate"),
        ("exact_recovery_rate", "Exact recovery rate"),
        ("mean_iterations", "Mean iterations"),
    ]
    centers = np.asarray(artifact["centers"], dtype=np.float64)
    widths = np.asarray(artifact["widths"], dtype=np.float64)
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
    for axis, (metric_key, label) in zip(axes.flat, metrics):
        values = np.asarray(artifact[metric_key], dtype=np.float64)
        vmin, vmax = dynamic_limits(values)
        image = axis.imshow(
            values,
            origin="lower",
            aspect="auto",
            extent=[float(centers.min()), float(centers.max()), float(widths.min()), float(widths.max())],
            cmap="viridis",
            vmin=vmin,
            vmax=vmax,
        )
        axis.set_title(label)
        axis.set_xlabel("interval center")
        axis.set_ylabel("interval width")
        fig.colorbar(image, ax=axis, shrink=0.85)
    fig.suptitle(title, fontsize=14)
    return fig


def edge_grid_table(artifact: dict[str, object]) -> pd.DataFrame:
    return pd.DataFrame(edge_weight_heatmap_rows(artifact)).sort_values(
        ["logical_success_rate", "convergence_rate", "exact_recovery_rate", "mean_iterations", "width", "center"],
        ascending=[False, False, False, True, True, True],
    ).reset_index(drop=True)


def edge_best_points_table(artifact: dict[str, object], top_k: int = 8) -> pd.DataFrame:
    return pd.DataFrame(best_edge_weight_rows(edge_weight_heatmap_rows(artifact), topk=top_k)).reset_index(drop=True)


code_data = load_css_code(npz_path)
hz_check_matrix = code_data["hz"]
lz_matrix = code_data["lz"]

case = enumerate_half_stabilizer_cases(code_data["problem"], max_cases=1)[0]
trace_interval = interval_from_center_width(trace_center, trace_width)
_supports_edge_weights = hasattr(
    relay_bp.MinSumBPDecoderTraceF64(
        code_data["problem"].hz,
        error_priors=code_data["problem"].error_priors,
        max_iter=6,
        alpha=1.0,
    ),
    "set_explicit_edge_message_weights",
)


def run_trace(case: dict[str, object], *, label: str, edge_weights: np.ndarray | None):
    tracer = make_min_sum_tracer(code_data["problem"], max_iter=max_iter, alpha=alpha, gamma0=None)
    snapshots = []
    tracer.reset()
    if edge_weights is not None:
        if not hasattr(tracer, "set_explicit_edge_message_weights"):
            raise RuntimeError("Rebuild relay_bp to expose explicit edge-message-weight bindings.")
        tracer.set_explicit_edge_message_weights(edge_weights)
    result = tracer.snapshot(case["syndrome"])
    snapshots.append(result)
    while tracer.current_iteration < result.max_iter and not result.success:
        result = tracer.run_iteration(case["syndrome"])
        snapshots.append(result)
    evaluation = evaluate_decode_result(result=result, error=case["error"], lz=code_data["problem"].lz)
    posterior_trace = np.vstack([np.asarray(snapshot.posterior_ratios, dtype=np.float64) for snapshot in snapshots])
    return {
        "label": label,
        "iterations": np.asarray([int(snapshot.iterations) for snapshot in snapshots]),
        "posterior_trace": posterior_trace,
        "final_success": bool(result.success),
        "logical_success": bool(evaluation.logical_success),
        "exact_recovery": bool(evaluation.exact_recovery),
        "final_iterations": int(result.iterations),
    }


trace_runs = [run_trace(case, label="plain BP", edge_weights=None)]
if _supports_edge_weights:
    same_weights = np.full(code_data["problem"].hz.nnz, trace_center, dtype=np.float64)
    trace_runs.append(run_trace(case, label=f"same edge weight {trace_center:.3f}", edge_weights=same_weights))
    for seed_offset in trace_seed_offsets:
        weights = code_capacity_common_module.build_edge_message_weights(
            code_data["problem"].hz,
            interval=trace_interval,
            coeff_seed=base_seed + int(seed_offset),
            support_bits=case["support_bits"],
        )
        trace_runs.append(run_trace(case, label=f"random edge weight seed {seed_offset}", edge_weights=weights))
else:
    print("Edge-weight trace needs a rebuilt relay_bp extension.")

trace_summary_df = pd.DataFrame([
    {
        "label": run["label"],
        "final_success": run["final_success"],
        "logical_success": run["logical_success"],
        "exact_recovery": run["exact_recovery"],
        "final_iterations": run["final_iterations"],
    }
    for run in trace_runs
])

assert len(trace_runs) >= 1


## Trace Summary


In [ ]:
display(trace_summary_df)


## Posterior Traces


In [ ]:
support_bits = list(case["support_bits"])
fig, axes = plt.subplots(len(trace_runs), 1, figsize=(10, 2.6 * len(trace_runs)), sharex=True, constrained_layout=True)
if len(trace_runs) == 1:
    axes = [axes]
for axis, run in zip(axes, trace_runs):
    trace = run["posterior_trace"][:, support_bits]
    for col_idx, bit in enumerate(support_bits):
        axis.plot(run["iterations"], trace[:, col_idx], marker="o", label=f"q{bit}")
    axis.set_title(run["label"])
    axis.set_ylabel("posterior ratio")
    axis.grid(alpha=0.25)
axes[-1].set_xlabel("iteration")
axes[0].legend(ncol=min(4, len(support_bits)), fontsize=8)
fig
